In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

TRAIN_PATH = "../input/train.csv"
TEST_PATH = "../input/test.csv"
SUBMISSION_NAME = "submissiontry_water.csv"

BATCH_SIZE = 256
EPOCHS = 200  # retained for reference, not used by RandomForest
LEARNING_RATE = 0.001

DATASET_SIZE = 500000




In [2]:
datatypes = {
    "key": "str",
    "fare_amount": "float32",
    "pickup_datetime": "str",
    "pickup_longitude": "float32",
    "pickup_latitude": "float32",
    "dropoff_longitude": "float32",
    "dropoff_latitude": "float32",
    "passenger_count": "uint8",
}
trainKaggle = pd.read_csv(
    TRAIN_PATH, nrows=DATASET_SIZE, dtype=datatypes, usecols=[1, 2, 3, 4, 5, 6, 7]
)
testKaggle = pd.read_csv(TEST_PATH)




In [3]:
train_df, test_df = train_test_split(trainKaggle, test_size=0.50, random_state=1)




In [4]:
train_df, validation_df = train_test_split(train_df, test_size=0.10, random_state=1)




In [5]:
print("testKaggle Size %d" % len(testKaggle))
print("train_df Size %d" % len(train_df))
print("validation_df Size %d" % len(validation_df))
print("test_df Size %d" % len(test_df))




testKaggle Size 9914
train_df Size 225000
validation_df Size 25000
test_df Size 250000


In [6]:
def remove_datapoints_from_water(df):
    """
    Original implementation attempted to load a remote mask image,
    which fails in the Kaggle environment. Here we simply return the dataframe
    unchanged, keeping all rows that passed the earlier geographic filters.
    """
    return df


def clean(df):
    print(" Old size: %d" % len(df))
    df = df.dropna(how="any", axis="rows")
    print(" New size after dropna: %d" % len(df))

    df = df[
        (df["dropoff_longitude"] != df["pickup_longitude"])
        & (df["dropoff_latitude"] != df["pickup_latitude"])
    ]
    print(" New size after removing same long lat: %d" % len(df))

    df = df[
        (df["dropoff_longitude"] != 0)
        & (df["pickup_longitude"] != 0)
        & (df["dropoff_latitude"] != 0)
        & (df["pickup_latitude"] != 0)
    ]
    print(" New size after removing 0 long lat: %d" % len(df))

    MinMax = (-74.5, -72.8, 40.5, 41.8)
    df = df[
        (MinMax[0] <= df["pickup_longitude"]) & (df["pickup_longitude"] <= MinMax[1])
    ]
    df = df[
        (MinMax[0] <= df["dropoff_longitude"]) & (df["dropoff_longitude"] <= MinMax[1])
    ]
    df = df[(MinMax[2] <= df["pickup_latitude"]) & (df["pickup_latitude"] <= MinMax[3])]
    df = df[
        (MinMax[2] <= df["dropoff_latitude"]) & (df["dropoff_latitude"] <= MinMax[3])
    ]
    print(" New size after only NYC: %d" % len(df))

    df = df[(0 < df["fare_amount"]) & (df["fare_amount"] <= 50)]
    print(" New size after removing outliers: %d" % len(df))

    df = df[(df["passenger_count"] > 0) & (df["passenger_count"] <= 6)]
    print(" New size after removing 6=>passenger_count > 0 : %d" % len(df))

    nyc_coord = (40.7141667, -74.0063889)
    fk_coord = (40.639722, -73.778889)
    ewr_coord = (40.6925, -74.168611)
    lga_coord = (40.77725, -73.872611)
    sol_coord = (40.6892, -74.0445)

    df = df[
        (nyc_coord[1] != df["pickup_longitude"])
        & (df["pickup_latitude"] != nyc_coord[0])
    ]
    df = df[
        (nyc_coord[1] != df["dropoff_longitude"])
        & (df["dropoff_latitude"] != nyc_coord[0])
    ]
    print(" New size after NY airport: %d" % len(df))

    df = df[
        (fk_coord[1] != df["pickup_longitude"]) & (df["pickup_latitude"] != fk_coord[0])
    ]
    df = df[
        (fk_coord[1] != df["dropoff_longitude"])
        & (df["dropoff_latitude"] != fk_coord[0])
    ]
    print(" New size after jfk airport: %d" % len(df))

    df = df[
        (ewr_coord[1] != df["pickup_longitude"])
        & (df["pickup_latitude"] != ewr_coord[0])
    ]
    df = df[
        (ewr_coord[1] != df["dropoff_longitude"])
        & (df["dropoff_latitude"] != ewr_coord[0])
    ]
    print(" New size after ewr airport: %d" % len(df))

    df = df[
        (lga_coord[1] != df["pickup_longitude"])
        & (df["pickup_latitude"] != lga_coord[0])
    ]
    df = df[
        (lga_coord[1] != df["dropoff_longitude"])
        & (df["dropoff_latitude"] != lga_coord[0])
    ]
    print(" New size after lgr airport: %d" % len(df))

    df = df[
        (sol_coord[1] != df["pickup_longitude"])
        & (df["pickup_latitude"] != sol_coord[0])
    ]
    df = df[
        (sol_coord[1] != df["dropoff_longitude"])
        & (df["dropoff_latitude"] != sol_coord[0])
    ]
    print(" New size after sol removed: %d" % len(df))

    print("Old size before water filter: %d" % len(df))
    df = remove_datapoints_from_water(df)
    print("New size after water filter: %d" % len(df))

    return df


def late_night(row):
    return 1 if (row["hour"] <= 3) else 0


def night(row):
    return 1 if ((row["hour"] > 20) and (row["weekday"] < 5)) else 0


def rush_hour(row):
    return 1 if ((16 <= row["hour"] <= 20) and (row["weekday"] < 5)) else 0


def manhattan(pickup_lat, pickup_long, dropoff_lat, dropoff_long):
    return np.abs(dropoff_lat - pickup_lat) + np.abs(dropoff_long - pickup_long)


def haversine_distance(lat1, lon1, lat2, lon2):
    p = np.pi / 180.0
    lat1, lon1, lat2, lon2 = [np.radians(x) for x in (lat1, lon1, lat2, lon2)]
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6371 * c
    return km


def add_time_features(df):
    """
    Vectorized creation of time‑based features, removing the costly df.apply calls.
    The resulting columns are identical to the original implementation.
    """
    df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], errors="coerce")
    df["year"] = df["pickup_datetime"].dt.year
    df["month"] = df["pickup_datetime"].dt.month
    df["day"] = df["pickup_datetime"].dt.day
    df["hour"] = df["pickup_datetime"].dt.hour
    df["weekday"] = df["pickup_datetime"].dt.weekday

    df["night"] = ((df["hour"] > 20) & (df["weekday"] < 5)).astype(int)
    df["late_night"] = (df["hour"] <= 3).astype(int)
    df["rush_hour"] = ((df["hour"].between(16, 20)) & (df["weekday"] < 5)).astype(int)

    df["pickup_datetime"] = df["pickup_datetime"].astype(str)
    return df


def add_coordinate_features(df):
    df["latdiff"] = df["pickup_latitude"] - df["dropoff_latitude"]
    df["londiff"] = df["pickup_longitude"] - df["dropoff_longitude"]
    return df


def add_distances_features(df):
    df["manhattan"] = manhattan(
        df["pickup_latitude"],
        df["pickup_longitude"],
        df["dropoff_latitude"],
        df["dropoff_longitude"],
    )
    df["distance"] = haversine_distance(
        df["pickup_latitude"],
        df["pickup_longitude"],
        df["dropoff_latitude"],
        df["dropoff_longitude"],
    )
    return df


def output_submission(raw_test, prediction, id_column, prediction_column, file_name):
    df = pd.DataFrame({prediction_column: prediction.ravel()})
    df[id_column] = raw_test[id_column].values
    df[[id_column, prediction_column]].to_csv(file_name, index=False)
    print("Output complete: {}".format(file_name))




In [7]:
print("train_df clean")
train_df = clean(train_df)
print("validation_df clean")
validation_df = clean(validation_df)
print("test_df clean")
test_df = clean(test_df)




train_df clean
 Old size: 225000
 New size after dropna: 224999
 New size after removing same long lat: 218269
 New size after removing 0 long lat: 217840
 New size after only NYC: 217568
 New size after removing outliers: 214926
 New size after removing 6=>passenger_count > 0 : 214138
 New size after NY airport: 214091
 New size after jfk airport: 214091
 New size after ewr airport: 214090
 New size after lgr airport: 214074
 New size after sol removed: 214074
Old size before water filter: 214074
New size after water filter: 214074
validation_df clean
 Old size: 25000
 New size after dropna: 25000
 New size after removing same long lat: 24281
 New size after removing 0 long lat: 24252
 New size after only NYC: 24231
 New size after removing outliers: 23916
 New size after removing 6=>passenger_count > 0 : 23824
 New size after NY airport: 23815
 New size after jfk airport: 23815
 New size after ewr airport: 23814
 New size after lgr airport: 23810
 New size after sol removed: 23810
Ol

In [8]:
train_df.describe()




,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,214074.000000,214074.000000,214074.000000,214074.000000,214074.000000,214074.000000
mean,10.722075,-73.976540,40.751564,-73.974907,40.751862,1.689360
std,7.709221,0.038899,0.035790,0.038904,0.039346,1.304919
min,0.010000,-74.408524,40.508606,-74.434807,40.517712,1.000000
25%,6.000000,-73.992310,40.737022,-73.991554,40.736248,1.000000
50%,8.500000,-73.982155,40.753597,-73.980637,40.754211,1.000000
75%,12.500000,-73.968872,40.767696,-73.965736,40.768509,2.000000
max,50.000000,-72.856972,41.692429,-72.854942,41.714634,6.000000


In [9]:
validation_df.describe()




,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,23810.000000,23810.000000,23810.000000,23810.000000,23810.000000,23810.000000
mean,10.760071,-73.976219,40.751434,-73.974716,40.751774,1.703612
std,7.810133,0.036036,0.028086,0.034318,0.030652,1.322457
min,0.010000,-74.222153,40.514938,-74.222618,40.505650,1.000000
25%,6.000000,-73.992386,40.736752,-73.991669,40.736540,1.000000
50%,8.500000,-73.982208,40.753796,-73.980675,40.754101,1.000000
75%,12.500000,-73.968647,40.767585,-73.965836,40.768409,2.000000
max,50.000000,-72.900436,41.366138,-72.848091,41.452766,6.000000


In [10]:
print("train_df add_time_features")
train_df = add_time_features(train_df)
print("validation_df add_time_features")
validation_df = add_time_features(validation_df)
print("test_df add_time_features")
test_df = add_time_features(test_df)
print("testKaggle add_time_features")
testKaggle = add_time_features(testKaggle)




train_df add_time_features
validation_df add_time_features
test_df add_time_features
testKaggle add_time_features


In [11]:
print("train_df add_coordinate_features")
train_df = add_coordinate_features(train_df)
print("validation_df add_coordinate_features")
validation_df = add_coordinate_features(validation_df)
print("test_df add_coordinate_features")
test_df = add_coordinate_features(test_df)
print("testKaggle add_coordinate_features")
testKaggle = add_coordinate_features(testKaggle)




train_df add_coordinate_features
validation_df add_coordinate_features
test_df add_coordinate_features
testKaggle add_coordinate_features


In [12]:
print("train_df add_distances_features")
train_df = add_distances_features(train_df)
print("validation_df add_distances_features")
validation_df = add_distances_features(validation_df)
print("test_df add_distances_features")
test_df = add_distances_features(test_df)
print("testKaggle add_distances_features")
testKaggle = add_distances_features(testKaggle)
print("Done with Adding features")




train_df add_distances_features
validation_df add_distances_features
test_df add_distances_features
testKaggle add_distances_features
Done with Adding features


In [13]:
dropped_columns = ["pickup_datetime"]
train_df = train_df.drop(dropped_columns, axis=1)
validation_df = validation_df.drop(dropped_columns, axis=1)
test_df = test_df.drop(dropped_columns, axis=1)
testKaggle_clean = testKaggle.drop(dropped_columns + ["key"], axis=1)
print("Done with dropped_columns")

train_labels = train_df["fare_amount"].values
validation_labels = validation_df["fare_amount"].values
test_labels = test_df["fare_amount"].values

train_labels_log = np.log1p(train_labels)
validation_labels_log = np.log1p(validation_labels)

# Drop the target column from feature sets (axis=1 ensures column removal)
train_df = train_df.drop(["fare_amount"], axis=1)
validation_df = validation_df.drop(["fare_amount"], axis=1)
test_df = test_df.drop(["fare_amount"], axis=1)
print("Done with Labels and log‑transform")




Done with dropped_columns
Done with Labels and log‑transform


In [14]:
print("train shape:", train_df.shape)
print("validation shape:", validation_df.shape)
print("test shape:", test_df.shape)




train shape: (214074, 17)
validation shape: (23810, 17)
test shape: (238034, 17)


In [15]:
scaler = preprocessing.MinMaxScaler()
train_df_scaled = scaler.fit_transform(train_df)
validation_df_scaled = scaler.transform(validation_df)
test_scaled = scaler.transform(test_df)
testKaggle_scaled = scaler.transform(testKaggle_clean)




In [16]:
rf_model = RandomForestRegressor(
    n_estimators=500,  # fewer trees -> faster training
    max_depth=30,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    min_samples_split=2,
)
rf_model.fit(train_df_scaled, train_labels_log)
print("RandomForest training complete on log‑target.")




RandomForest training complete on log‑target.


In [17]:
def plot_loss_accuracy_rmse(*args, **kwargs):
    pass


plot_loss_accuracy_rmse()




In [18]:
val_pred_log = rf_model.predict(validation_df_scaled)
validation_pred = np.expm1(val_pred_log)  # back to original scale

prediction_log = rf_model.predict(test_scaled)
prediction = np.expm1(prediction_log)  # back to original scale

predictionKaggle_log = rf_model.predict(testKaggle_scaled)
predictionKaggle = np.expm1(predictionKaggle_log)




In [19]:
test_rmse = np.sqrt(np.mean((test_labels - prediction) ** 2))
validation_rmse = np.sqrt(np.mean((validation_labels - validation_pred) ** 2))
print(f"Test RMSE: {test_rmse:.5f}")
print(f"Validation RMSE: {validation_rmse:.5f}")




Test RMSE: 2.85348
Validation RMSE: 2.83415


In [20]:
output_submission(testKaggle, predictionKaggle, "key", "fare_amount", SUBMISSION_NAME)




Output complete: submissiontry_water.csv


In [21]:
print("First prediction example:")
print(prediction[0])
print("Corresponding test label (for reference, not used in submission):")
print(test_labels[0])

First prediction example:
5.356011989571693
Corresponding test label (for reference, not used in submission):
4.5
